# Development of Arctic boundary current model

See `gridap_development/ArcticBoundaryCurrentModel12.ipynb` for development of NS solver using `gridap`. This version switches to `nuPGCM`.

13.  `ArcticBoundaryCurrentModel13.ipynb`: Migrate to [`nuPGCM`](https://github.com/hgpeterson/nuPGCM) by merging `ArcticBoundaryCurrentModel12.ipynb` with `nuPGCM/examples.run.jl`.
14.  `ArcticBoundaryCurrentModel14.ipynb`: Switch to `ArcticBasin.msh` geometry.
15.  `ArcticBoundaryCurrentModel15.ipynb`: Make consistent with `examples/run.jl`.
16.  `ArcticBoundaryCurrentModel16.ipynb`: Adjust plotting. Experiment with different parameter settings.
17.  `ArcticBoundaryCurrentModel17.ipynb`: Add forcing from v0.4.0 and update from my customizations (0.3.0_custom).
18.  `ArcticBoundaryCurrentModel18.ipynb`: Add wind forcing.
19.  `ArcticBoundaryCurrentModel19.ipynb`: Add buoyancy forcing.
20.  `ArcticBoundaryCurrentModel20.ipynb`: Experiment with Nøst & Isachsen (2003) Appendix B configuration

Goals: Add T/S equation of state

Run this notebook with `nuPGCM` branch `0.4.0_custom`

twnh August, September '25

## Problem statement

Following Peterson & Callies (2025, in review at JPO, their (A1)–(A4), (13)), the (non-dimensional) equations we solve are:
$$
\text{Inversion: } 
\left\lbrace
\begin{aligned}
-v &= - \frac{\partial p}{\partial x} + \epsilon^2 \left( \alpha^2 \frac{\partial ^2 u}{\partial x^2} + \alpha^2 \frac{\partial ^2 u}{\partial y^2} + \frac{\partial ^2 u}{\partial z^2}\right), \\
u &= - \frac{\partial p}{\partial y} + \epsilon^2 \left( \alpha^2 \frac{\partial ^2 v}{\partial x^2} + \alpha^2 \frac{\partial ^2 v}{\partial y^2} + \frac{\partial ^2 v}{\partial z^2}\right), \\
\frac{\partial p}{\partial z} & = b + \epsilon^2 \alpha^2 \left( \alpha^2 \frac{\partial ^2 w}{\partial x^2} + \alpha^2 \frac{\partial ^2 w}{\partial y^2} + \frac{\partial ^2 w}{\partial z^2} \right) ,\\
\nabla\cdot \boldsymbol{u} &= 0,
\end{aligned}
\right.
$$
$$
\text{Buoyancy: } 
\mu \varrho \left( \frac{\partial b}{\partial t} + \boldsymbol{u} \cdot \nabla b \right) = \epsilon^2 \alpha^2 \left[ \frac{\partial}{\partial x} \left( \kappa(\boldsymbol{x}) \frac{\partial b}{\partial x}\right) + \frac{\partial}{\partial y} \left( \kappa (\boldsymbol{x}) \frac{\partial b}{\partial y}\right) + \frac{\partial}{\partial z} \left( \kappa(\boldsymbol{x}) \frac{\partial b}{\partial z}\right) \right]
$$
(see also Peterson & Callies, 2025, in prep. for JAMES, their (12)–(14)).
Forcing is given by:
$$
\text{Boundary conditions: }\left\lbrace
\begin{aligned}
\boldsymbol{t} \cdot \boldsymbol{\sigma} \cdot \mathbf{n} = \tau_{\text{imposed}} (\boldsymbol{x}) &\text{ on } \Gamma_s, \\
\boldsymbol{u} = \boldsymbol{0} &\text{ on } \Gamma_w . \\
\end{aligned}
\right.
$$
However, their solutions are unforced, and the `examples/run.jl` code has no forcing.
The domain $\Omega$ rotates with constant Coriolis parameter and has boundary $\partial \Omega$, ${\mathbf n}$ is the unit outward normal, $\boldsymbol{t}$ is the tangential direction at the surface, $\Gamma_s$, and $\boldsymbol{\sigma} = \nabla u$ is the stress vector. The driving force is the tangential stress $\tau_{\text{imposed}}$ (units of $\text{m}^{2} \text{s}^{-2}$ ). The mean value of the pressure anomaly is constrained to equal zero,
$$
\int_\Omega p \ {\rm d}\Omega = 0 .
$$
The initial condition is no flow and $b_{tot} = b + N^2 z$, so that $b$ measures the anomaly from uniform stratification (see: `nuPGCM/src/model.jl` and this [issue](https://github.com/hgpeterson/nuPGCM/issues/5).)

We're also interested in:
1. Two-component equation of state for buoyancy $b$.
2. Buoyancy forcing on $\Gamma_s$ and $\Gamma_w$.
3. Free-slip (no stress) boundary conditions on $\Gamma_w$ (but not for the stress-driven case, unless the integrated applied stress vanishes).

### Non-dimensional parameters

The scaling is as follows (Peterson & Callies, 2025, in review):
$$
\begin{align}
x, y & \sim L, \nonumber \\
z & \sim H_0, \nonumber \\
u, v & \sim U_0, \nonumber \\
w & \sim \frac{U_0 H_0}{L}, \nonumber \\
f & \sim f_0, \nonumber \\
\nu & \sim \nu_0, \nonumber \\
\kappa & \sim \kappa_0 ,\nonumber \\
p & \sim f_0 U_0 L, \text{ with units of Pa/(kgm$^{-3}$)}\nonumber \\
b & \sim \frac{f_0 U_0 L}{H_0} = N^2 H_0  \implies U_0 = \frac{N^2 H_0^2}{f_0 L} , \nonumber \\
t & \sim \frac{L}{U_0} , \nonumber \\
\tau \text{ (units m}^2\text{s}^2\text{)} & \sim \frac{\nu_0 U_0}{H_0} , \nonumber 
\end{align}
$$
with non-dimensional numbers
$$
\begin{align}
\text{Ekman number: } \epsilon & = \sqrt{\frac{\nu_0} {f_0 H_0^2}} , \nonumber \\
\text{Burger number: } \varrho & = \frac{N^2 H_0^2} {f_0^2 L^2} ,  \nonumber \\
\text{Turbulent Prandtl number: } \mu & = \frac{\nu_0} {\kappa_0} ,  \nonumber \\
\text{Aspect ratio: } \alpha & = \frac{H_0} {L}
\end{align}
$$

The dimensional viscosity is a constant $\nu = \nu_0$, but the diffusivity varies in space, usually decaying away from the bottom $\kappa(z)$.

We specify:
\begin{align}
\text{Characteristic domain depth:  } & H_0, \nonumber\\
\text{Characteristic domain width:  } & L, \nonumber\\
\text{Coriolis parameter (uniform): } & f_0,\nonumber\\
\text{Characteristic diffusivity:   } & \kappa_0, \nonumber\\
\text{Characteristic viscosity (uniform): } & \nu_0, \nonumber\\
\text{Characteristic buoyancy frequency: } & N,  \nonumber\\
%\text{Characteristic surface stress:   } & \tau_0, \nonumber\\
\text{Non-dimensional diffusivity function: } & \kappa (\boldsymbol{x}), \nonumber\\
\text{Non-dimensional domain depth function: } & H (\boldsymbol{x}), \nonumber\\
%\text{Non-dimensional surface stress function: } & \tau (\boldsymbol{x}), \nonumber\\
\text{Initial non-dimensional buoyancy field: } & b_0 (\boldsymbol{x}), \nonumber\\
\end{align}

#### Import packages

In [ ]:
using nuPGCM
using JLD2
using LinearAlgebra
using Printf
using Revise
using PyPlot

#### Configure nuPGCM

In [ ]:
set_out_dir!(joinpath(@__DIR__, ""))
arch = CPU()
force_build_inversion = true
force_build_evolution = true 
pygui(false)
plt.style.use(joinpath(@__DIR__, "plots.mplstyle"))
plt.close("all")

#### Define physical parameters

In [ ]:
# Realistic parameters
# L  = 1000.0e3   # (m) domain size
# H₀ = 1000.0     # (m) domain height
# f₀ = 2.0e-4     # (s⁻¹) Coriolis parameter
# ν₀ = 8.0e-2     # (m²/s) kinematic viscosity
# κ₀ = 8.0e-2     # (m²/s) kinematic diffusivity
# N² = (10*f₀)^2  # (s⁻²) buoyancy frequency squared

# Values consistent with examples/run.jl
L  = 1000.0e3               # (m) domain size
H₀ = L/2                    # (m) domain height
f₀ = 2.0e-4                 # (s⁻¹) Coriolis parameter
β  = 0.0                    # (m⁻¹s⁻¹) beta parameter. Set to zero for an f-plane. Non-zero values require a 3D mesh.
ν₀ = 0.1^2 * f₀ * H₀^2      # (m²/s) kinematic viscosity. Effectively set the Ekman number here with the choice of ν₀. examples/run.jl uses ν₀ = 0.2^2 * f₀ * H₀^2
κ₀ = ν₀                     # (m²/s) kinematic diffusivity
N² = (10*f₀)^2              # (s⁻²) buoyancy frequency squared
# N² = (f₀)^2              # (s⁻²) buoyancy frequency squared

println("Domain horizontal size            : ", L, " m")
println("Domain vertical size              : ", H₀, " m")
println("Coriolis parameter                : ", f₀, " s⁻¹")
println("Beta parameter                    : ", β, " m⁻¹s⁻¹")
println("Kinematic viscosity               : ", ν₀, " m⁻²s⁻¹")
println("Kinematic diffusivity             : ", κ₀, " m⁻²s⁻¹")
println("Buoyancy frequency                : ", sqrt(N²), " s⁻¹\n")

#### Wind forcing parameters

In [ ]:
windspeed = 0.0             # m/s, maximum wind speed at r = L
# windspeed = -300.0          # m/s, maximum wind speed at r = L
println("Maximum windspeed                 : ", windspeed, "ms⁻¹")
ρₒ = 1000.0  # (kg/m³) reference density
cᴰ = 2.5e-3  # dimensionless drag coefficient
ρₐ = 1.225   # (kg/m³) average density of air
function u₁₀(x_in,windspeed)
    x, y = x_in[1], x_in[2]
    r = sqrt(x^2 + y^2)
    θ = atan(y, x)
    radial_wind = windspeed * r
    u₁₀ = radial_wind * [ - sin(θ),  cos(θ), 0.0 ] # m s⁻¹
    return u₁₀
end
τ₀(x) = Vector((ρₐ / ρₒ) * cᴰ * u₁₀(x,windspeed) * norm(u₁₀(x,windspeed))) # m² s⁻²
τˣ(x) = τ₀(x)[1]/τₛ  # non-dimensional zonal wind stress
τʸ(x) = τ₀(x)[2]/τₛ  # non-dimensional meridional wind stress
uₛ = sqrt((ρₐ / ρₒ) * cᴰ) # m/s, friction velocity  
println("Friction velocity                 : ", uₛ, " ms⁻¹")
# U₀ = uₛ^2 * H₀ / ν₀   # (m/s) speed scale

#### Buoyancy initial condition

This, implicitly, defines the buoyancy boundary condition at the surface too.

In [ ]:
# ΔT = -100.0                              # K, surface temperature difference across the front
ΔT = 0.0                              # K, surface temperature difference across the front
println("Surface temperature diff.         : ", ΔT, " K")
αₜ  = 2.0e-4                            # K⁻¹, thermal expansion coefficient
g  = 9.81                               # m/s², gravitational acceleration
b⁺ =  g * αₜ * (ΔT / 2.0) / (N² * H₀)    # (non-dim) surface buoyancy on the warm side of the front
b⁻ = -g * αₜ * (ΔT / 2.0) / (N² * H₀)    # (non-dim) surface buoyancy on the cold side of the front
r₀ = 1 / 2.0                            # (non-dim) radius of the buoyancy front
Δr = 1 / 8.0                            # (non-dim) width of the buoyancy front
z₀ = 1 / 4.0                            # (non-dim) e-folding depth scale for buoyancy decay
function initial_b(x_in,b⁺,b⁻,r₀,Δr,z₀) # Function to set the initial buoyancy anomaly field, and hence the buoyancy boundary condition (Dirichlet bc)
    x, y, z = x_in[1], x_in[2], x_in[3]
    r = sqrt(x^2 + y^2)
    b = b⁻ + (b⁺ - b⁻) * 0.5 * (1.0 + tanh((r - r₀) / Δr))
    b = b * exp(z/z₀)  # make buoyancy decay exponentially with depth
    return b
end
b₀(x) = initial_b(x,b⁺,b⁻,r₀,Δr,z₀)     # surface buoyancy boundary condition
println("Surface buoyancy diff.            : ", b⁺ - b⁻, " m²s⁻²\n")
println("Buoyancy forcing scale            : ", N² * H₀, " m²s⁻²")
# U₀ = uₛ^2 * H₀ / ν₀   # (m/s) speed scale based in stress

#### Compute and report non-dimensional parameters

In [ ]:
U₀ = N² * H₀^2 / (f₀ * L)   # (m/s) speed scale based on thermal wind
println("Speed scale                       : ", U₀, " ms⁻¹")
τₛ = ν₀ * U₀ / H₀         # m²/s², stress scale
println("Stress scale                      : ", τₛ, " m²s⁻²")
println("Time scale                        : ", L/U₀, " s")
println("Pressure scale                    : ", f₀*U₀*L*ρₒ, " Pa")

Ekman_layer_depth = sqrt(2*ν₀ / f₀) # (m), Ekman layer depth
println("Ekman layer depth                 : ", Ekman_layer_depth, " m")
println("Non-dimensional Ekman layer depth : ", Ekman_layer_depth/H₀)

ε = sqrt(ν₀ / (f₀ * H₀^2))      # Ekman number
println("\nEkman number ε                    : ", ε)
ϱ = (N² * H₀^2) / (f₀^2 * L^2)  # Burger number
println("Burger number ϱ                   : ", ϱ)
μ = ν₀ / κ₀                     # Turbulent Prandtl number
println("Turbulent Prandtl number μ        : ", μ)
μϱ = μ * ϱ  
println("Prandtl x Burger number μϱ        : ", μϱ)
α = H₀ / L                      # Aspect ratio
println("Domain aspect ratio α             : ", α)

#### Compute numerical parameters

In [ ]:
Δt = 1e-5*μϱ/ε^2/α^2            # Time step size
println("Non-dim. time step size           : ", Δt)

# Non-dimensional Coriolis parameter
f(x) = 1 + β*x[2]

# Final time of integration
Tf = 0.1*μϱ/ε^2  # simulation time. 
# Tf = 0.01*μϱ/ε^2  # simulation time. For testing.
# Tf = Δt           # Just one step.
println("Non-dim. final time               : ", Tf)
println("Number of steps                   : ", Tf/Δt,"\n")

### Build and load mesh

Select the mesh you want here.

In [ ]:
if(false)        # Use NI03 exponential bowl 2D mesh generation function depth(x), which is a template for a generic depth function.
    dim = 2
    hh = 1e-2   # Sets mesh spacing. Default setting.
    include("meshes/mesh_exp_bowl2D.jl")
    mesh_name = @sprintf("exp_bowl2D_%e_%e", hh, α)
    generate_exp_bowl_mesh_2D(hh, depth; savefile=mesh_name, DEBUG=false, show_gui=false)
elseif(false)   # Use Tom's 2D mesh generation function.
    include("meshes/mesh_ArcticBasin2D.jl")
    mesh_name = "ArcticBasin2D"
    dim = 2
    # hh = 2e-2   # Sets mesh spacing. Default setting.
    hh = 1e-2   # Sets mesh spacing
    H, dH, R, r = generate_arctic_basin2D( hh, α; show_gui=false, savefile=mesh_name, DEBUG=false)
elseif(false)   # Use Tom's 3D mesh generation function.
                # This option takes a while to run (like an hour on my MacBook), and it gives very similar results to the 2D case.
    include("meshes/mesh_ArcticBasin3D.jl")
    mesh_name = "ArcticBasin3D"
    dim = 3
    hh = 8e-2   # Sets mesh spacing
    H, dH, R, r = generate_arctic_basin3D( hh, α; show_gui=false, savefile=mesh_name, DEBUG=false)
elseif(true)   # Use Henry's original 2D parabolic domain and mesh, modified with Tom's tags in the new spaces.jl file in nuPGCM/src.
    dim = 2
    hh = 1e-2   # Sets mesh spacing
    include("meshes/mesh_bowl2D.jl")
    mesh_name = @sprintf("bowl2D_%e_%e", hh, α)
elseif(false)   # Use Henry's original 3D parabolic domain and mesh, modified with Tom's tags in the new spaces.jl file in nuPGCM/src.
                # This option takes a while to run, and it gives very similar results to the 2D case.
    dim = 3
    hh = 8e-2   # Sets mesh spacing
    include("meshes/mesh_bowl3D.jl")
    mesh_name = @sprintf("bowl%dD_%e_%e", dim, hh, α)
end # if
@info "Loading mesh from file: " mesh_name
mesh = Mesh(joinpath(@__DIR__, "meshes/$mesh_name.msh")) ;

#### Specify viscosity, diffusivity, and build `params` variable.

In [ ]:
@warn "Make sure the mesh geometry is consistent with the depth(x) definition."
# Non-dimensional diffusivity (enhanced at the bottom). See Peterson & Callies (2025) eq. (17)
mixing_depth = 0.1*α    # Depth over which diffusivity increases near the bottom. examples/run.jl uses mixing_depth = 0.1*α
# mixing_depth = 0.05*α    # Depth over which diffusivity increases near the bottom. examples/run.jl uses mixing_depth = 0.1*α
# mixing_depth = α    # Depth over which diffusivity increases near the bottom. examples/run.jl uses mixing_depth = 0.1*α
@info "Mixing depth (non-dimensional): " mixing_depth
κ(x) = 1e-2 + exp(-(x[3] + depth(x))/mixing_depth) ;
ν(x) = 1                # viscosity. examples/run.jl uses ν(x) = 1
params = Parameters(ε, α, μϱ, N²/(N² * α), Δt) ;       # Non-dimensional parameters, including N²/N² for consistency with nuPGCM, which is entirely non-dimensional.

### Build system



#### FE spaces

In [ ]:
spaces = Spaces(mesh, b₀)
fe_data = FEData(mesh, spaces)
@info "DOFs: $(fe_data.dofs.nu + fe_data.dofs.nv + fe_data.dofs.nw + fe_data.dofs.np)"

#### Build inversion matrices if necessary: `A_inversion`, `B_inversion`

To be sure, delete the saved `.jld2` files. Sometimes the file exists, but for a different parameter set, which leads to inconsistencies and errors.

In [ ]:
if !isdir(joinpath(@__DIR__, "matrices"))
    @info "Creating matrices directory"
    mkdir(joinpath(@__DIR__, "matrices"))
end
A_inversion_fname = joinpath(@__DIR__, @sprintf("matrices/A_inversion_%sD_%e_%e_%e_%e_%e.jld2", dim, hh, ε, α, f₀, β))

if force_build_inversion
    @warn "You set `force_build_inversion` to `true`, building matrices..."
    A_inversion, B_inversion, b_inversion = build_inversion_matrices(fe_data, params, f, ν, τˣ, τʸ; A_inversion_ofile=A_inversion_fname)
elseif !isfile(A_inversion_fname)
    @warn "A_inversion file not found, generating..."
    A_inversion, B_inversion, b_inversion = build_inversion_matrices(fe_data, params, f, ν, τˣ, τʸ; A_inversion_ofile=A_inversion_fname)
else
    file = jldopen(A_inversion_fname, "r")
    A_inversion = file["A_inversion"]
    close(file)
    B_inversion = nuPGCM.build_B_inversion(fe_data, params)
    b_inversion = nuPGCM.build_b_inversion(fe_data, params, τˣ, τʸ)
end
nothing

#### Re-order dofs

In [ ]:
A_inversion = A_inversion[fe_data.dofs.p_inversion, fe_data.dofs.p_inversion]
B_inversion = B_inversion[fe_data.dofs.p_inversion, :]
b_inversion = b_inversion[fe_data.dofs.p_inversion] ;

#### Preconditioner `P_inversion`

In [ ]:
if typeof(arch) == CPU
    @time "lu(A_inversion)" P_inversion = lu(A_inversion)
else
    P_inversion = Diagonal(on_architecture(arch, 1/hh^dim*ones(size(A_inversion, 1))))
end
nothing

#### Move matrices to architecture

In [ ]:
A_inversion = on_architecture(arch, A_inversion)
B_inversion = on_architecture(arch, B_inversion)
b_inversion = on_architecture(arch, b_inversion) ;

#### Setup inversion toolkit `inversion_toolkilt`

In [ ]:
inversion_toolkit = InversionToolkit(A_inversion, P_inversion, B_inversion, b_inversion; atol=1e-6, rtol=1e-6) ;

#### Build evolution matrices `A_adv`, `A_diff`, `B_diff`, `b_diff` and test against saved matrices

In [ ]:
A_adv, A_diff, B_diff, b_diff = build_evolution_system(fe_data, params, κ;
                                    force_build=force_build_evolution,
                                    filename=joinpath(@__DIR__, "matrices/evolution_$mesh_name.jld2")) ;

#### Re-order dofs

In [ ]:
A_adv  =  A_adv[fe_data.dofs.p_b, fe_data.dofs.p_b]
A_diff = A_diff[fe_data.dofs.p_b, fe_data.dofs.p_b]
B_diff = B_diff[fe_data.dofs.p_b, :]
b_diff = b_diff[fe_data.dofs.p_b] ;

#### Preconditioners `P_diff` and `P_adv`

In [ ]:
if typeof(arch) == CPU 
    P_diff = lu(A_diff)
    P_adv  = lu(A_adv)
else
    P_diff = Diagonal(on_architecture(arch, Vector(1 ./ diag(A_diff))))
    P_adv  = Diagonal(on_architecture(arch, Vector(1 ./ diag(A_adv))))
end
nothing

#### Move to architecture

In [ ]:
A_adv  = on_architecture(arch, A_adv)
A_diff = on_architecture(arch, A_diff)
B_diff = on_architecture(arch, B_diff)
b_diff = on_architecture(arch, b_diff) ;

#### Setup evolution toolkit `evolution_toolkit`

In [ ]:
evolution_toolkit = EvolutionToolkit(A_adv, P_adv, A_diff, P_diff, B_diff, b_diff) ;

#### Put it all together in the `model` struct

In [ ]:
model = rest_state_model(arch, params, fe_data, inversion_toolkit, evolution_toolkit) ;

#### Set initial buoyancy

In [ ]:
set_b!(model, x->b₀(x))
invert!(model) # sync flow with initial condition
save_vtk(model, ofile=@sprintf("%s/data/state_%016d.vtu", out_dir, 0))

### Solve

In [ ]:
n_steps = Int(round(Tf / Δt))
n_save = n_steps ÷ 100
# @time run!(model; n_steps=n_steps, n_save=n_save, n_plot=n_steps)
@time run!(model; n_steps=n_steps, n_save=n_save, n_plot=n_save)
println("Done.")